# Fine-tuning Llama 3.2 3B para Text-to-SQL con LoRA

Este notebook entrena un modelo especializado en generar consultas SQL usando **LoRA** sobre **Llama 3.2 3B**.

## ¿Qué haremos?
- Cargar Llama 3.2 3B Instruct
- Aplicar LoRA para fine-tuning eficiente
- Entrenar con datos de text-to-SQL
- Evaluar y guardar el modelo

## Requisitos:
- GPU con 12GB+ VRAM (Kaggle T4/P100)
- Cuenta Hugging Face
- Python 3.8+

## 1. Instalación de Dependencias

In [ ]:
# Instalar todas las dependencias necesarias
!pip install transformers datasets accelerate peft bitsandbytes torch huggingface_hub pandas numpy trl
!pip install ipywidgets

print("✅ Dependencias instaladas")

## 2. Autenticación Hugging Face

In [ ]:
from huggingface_hub import login

# Login en Hugging Face (necesario para Llama)
login()

print("✅ Autenticado en Hugging Face")

✅ Autenticado en Hugging Face


## 3. Importación de Librerías

In [17]:
import torch
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

# Transformers
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)

# LoRA y PEFT
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training,
    TaskType
)

# Datasets y entrenamiento
from datasets import Dataset, load_dataset
from trl import SFTTrainer

print(f"✅ Librerías importadas")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"💾 CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🎮 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

✅ Librerías importadas
🔥 PyTorch: 2.7.1+cpu
💾 CUDA disponible: False


In [ ]:
# Verificar recursos disponibles en Kaggle
print("🔍 VERIFICANDO RECURSOS KAGGLE")
print("=" * 40)

# Información del sistema
import psutil
print(f"💾 RAM Total: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"💾 RAM Disponible: {psutil.virtual_memory().available / 1024**3:.1f} GB")
print(f"🖥️ CPUs: {psutil.cpu_count()}")

# GPU info si está disponible
if torch.cuda.is_available():
    print(f"\n🎮 GPU DETECTADA:")
    for i in range(torch.cuda.device_count()):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"   GPU {i}: {gpu_name}")
        print(f"   VRAM: {gpu_memory:.1f} GB")
        
        # Memoria disponible
        torch.cuda.empty_cache()
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        cached = torch.cuda.memory_reserved(i) / 1024**3
        print(f"   Allocated: {allocated:.1f} GB")
        print(f"   Cached: {cached:.1f} GB")
        print(f"   Free: {gpu_memory - allocated:.1f} GB")
else:
    print("\n⚠️ NO GPU DETECTADA - Se usará CPU")
    print("   Recomendación: Activar GPU en Kaggle Settings")

# Disk space
disk = psutil.disk_usage('/')
print(f"\n💿 Disk Space:")
print(f"   Total: {disk.total / 1024**3:.1f} GB")
print(f"   Free: {disk.free / 1024**3:.1f} GB")

print("\n✅ Verificación completada")

## 4. Configuración del Modelo y Entrenamiento

Configuración optimizada para **Llama 3.2 3B** y **text-to-SQL**:

In [ ]:
# ====== CONFIGURACIÓN PRINCIPAL ======
CONFIG = {
    # Modelo Llama 3.2 3B
    "model_name": "meta-llama/Llama-3.2-3B-Instruct",
    
    # Dataset
    "dataset_name": "gretelai/synthetic_text_to_sql",
    "num_samples": 800,  # Más muestras para aprovechar el modelo 3B
    
    # Directorios
    "output_dir": "../models/llama-sql-lora",
    "logs_dir": "../logs",
    
    # Parámetros de entrenamiento optimizados para Kaggle
    "max_seq_length": 512,   # Secuencias más largas para 3B
    "batch_size": 1,         # Batch pequeño por memoria en Kaggle
    "gradient_accumulation": 8, # Simula batch_size = 8
    "learning_rate": 2e-4,   # LR más alto para 3B
    "num_epochs": 3,         # Más epochs para aprovechar el modelo
    "warmup_ratio": 0.05,    # Warmup más corto
    "save_steps": 50,        # Guardar más frecuente
    "eval_steps": 50,
    "eval_strategy": "steps",
    "save_strategy": "epoch",
    "logging_steps": 5,
    
    # Parámetros de evaluación
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
}

# ====== CONFIGURACIÓN LORA PARA LLAMA 3.2 3B ======
LORA_CONFIG = {
    "r": 16,                  # Rango más alto para modelo 3B
    "lora_alpha": 32,         # Alpha proporcional (2x rank)
    "lora_dropout": 0.05,     # Dropout menor para modelo más grande
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM,
    
    # Módulos específicos de Llama 3.2 para SQL - más módulos para 3B
    "target_modules": [
        "q_proj", "k_proj", "v_proj"
    ]
}

# Crear directorios
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["logs_dir"], exist_ok=True)

print("✅ Configuración establecida")
print(f"📦 Modelo: {CONFIG['model_name']}")
print(f"📊 Muestras: {CONFIG['num_samples']}")
print(f"🎯 LoRA rank: {LORA_CONFIG['r']}")
print(f"📁 Output: {CONFIG['output_dir']}")

✅ Configuración establecida
📦 Modelo: meta-llama/Llama-3.2-1B-Instruct
📊 Muestras: 100
🎯 LoRA rank: 8
📁 Output: ../models/llama-sql-lora


## 📋 Configuración Específica para Kaggle

### ⚙️ Configuración óptima para Kaggle:
- **Modelo**: Llama 3.2 3B (más potente que 1B)
- **GPU**: Aprovecha T4/P100 de Kaggle
- **Memoria**: Optimizada para ~16GB RAM
- **Batch size**: 1 con gradient_accumulation=8 (simula batch=8)
- **Secuencias**: 512 tokens (más contexto)
- **LoRA rank**: 16 (más parámetros entrenables)

### 🚀 Tiempo estimado en Kaggle:
- **Con GPU T4**: 2-4 horas
- **Con GPU P100**: 1.5-3 horas
- **Con CPU**: 8-12 horas (no recomendado)

### 💡 Consejos para Kaggle:
1. Activa GPU en Settings
2. Usa "Internet" para descargar el modelo
3. Guarda checkpoints frecuentes (save_steps=50)
4. Monitorea memoria con `!nvidia-smi`

## 5. Carga y Filtrado del Dataset

In [ ]:
def cargar_y_filtrar_datos():
    """Carga y filtra el dataset de SQL"""
    print("📥 Cargando dataset de text-to-SQL...")
    
    # Cargar más muestras para poder hacer split
    total_samples = int(CONFIG["num_samples"] * 1.25)  # 25% extra para eval
    dataset = load_dataset(
        CONFIG["dataset_name"], 
        split=f"train[:{total_samples}]"
    )
    df = pd.DataFrame(dataset)
    
    print(f"📊 Dataset original: {len(df)} ejemplos")
    
    # Filtros de calidad para SQL
    print("🔍 Aplicando filtros...")
    
    # 1. SQL válido y no vacío
    df = df[df['sql'].notna() & (df['sql'].str.len() > 15)]
    print(f"   Después filtro SQL válido: {len(df)}")
    
    # 2. Longitud razonable (evitar SQL extremadamente largos)
    df = df[df['sql'].str.len() < 800]
    print(f"   Después filtro longitud: {len(df)}")
    
    # 3. Pregunta válida
    df = df[df['sql_prompt'].notna() & (df['sql_prompt'].str.len() > 10)]
    print(f"   Después filtro pregunta: {len(df)}")
    
    # 4. Contexto/esquema válido
    df = df[df['sql_context'].notna() & (df['sql_context'].str.len() > 20)]
    print(f"   Después filtro contexto: {len(df)}")
    
    # 5. Sin errores obvios
    df = df[~df['sql'].str.contains('ERROR|error|undefined', case=False, na=False)]
    print(f"   Después filtro errores: {len(df)}")
    
    print(f"\n✅ Dataset final: {len(df)} ejemplos de calidad")
    
    # Crear split train/eval (80/20)
    train_size = CONFIG["num_samples"]
    train_df = df.head(train_size)
    eval_df = df.tail(len(df) - train_size)
    
    print(f"📊 Split creado: {len(train_df)} train, {len(eval_df)} eval")
    
    return train_df, eval_df

# Cargar datos
train_df, eval_df = cargar_y_filtrar_datos()

# Mostrar estadísticas
print(f"\n📈 Estadísticas:")
print(f"   SQL promedio: {train_df['sql'].str.len().mean():.0f} caracteres")
print(f"   Pregunta promedio: {train_df['sql_prompt'].str.len().mean():.0f} caracteres")

# Mostrar ejemplo
print(f"\n📝 Ejemplo:")
ejemplo = train_df.iloc[0]
print(f"Pregunta: {ejemplo['sql_prompt'][:100]}...")
print(f"SQL: {ejemplo['sql']}")

📥 Cargando dataset de text-to-SQL...
📊 Dataset original: 100 ejemplos
🔍 Aplicando filtros...
   Después filtro SQL válido: 100
   Después filtro longitud: 100
   Después filtro pregunta: 100
   Después filtro contexto: 100
   Después filtro errores: 100

✅ Dataset final: 100 ejemplos de calidad

📈 Estadísticas:
   SQL promedio: 121 caracteres
   Pregunta promedio: 83 caracteres

📝 Ejemplo:
Pregunta: What is the total volume of timber sold by each salesperson, sorted by salesperson?...
SQL: SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;


## 6. Formateo de Datos para Llama 3.2

In [ ]:
def formatear_para_llama32(df):
    """Formatea los datos usando el template de Llama 3.2"""
    print("🔄 Formateando datos para Llama 3.2...")
    
    # Template optimizado para text-to-SQL
    TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
{schema}

Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{sql}<|eot_id|>"""
    
    formatted_data = []
    
    for _, row in df.iterrows():
        # Crear texto formateado
        text = TEMPLATE.format(
            schema=row['sql_context'].strip(),
            question=row['sql_prompt'].strip(),
            sql=row['sql'].strip()
        )
        
        formatted_data.append({"text": text})
    
    print(f"✅ {len(formatted_data)} ejemplos formateados")
    
    # Mostrar ejemplo formateado
    print(f"\n📝 Ejemplo formateado:")
    print(formatted_data[0]["text"][:500] + "...")
    
    return formatted_data

# Formatear datos
training_data = formatear_para_llama32(train_df)
eval_data = formatear_para_llama32(eval_df)

# Crear datasets
train_dataset = Dataset.from_list(training_data)
eval_dataset = Dataset.from_list(eval_data)
print(f"\n📦 Datasets creados: {len(train_dataset)} train, {len(eval_dataset)} eval")

🔄 Formateando datos para Llama 3.2...
✅ 100 ejemplos formateados

📝 Ejemplo formateado:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE...

📦 Dataset creado: 100 ejemplos


## 7. Carga del Modelo Llama 3.2 3B

In [ ]:
def cargar_modelo_llama32():
    """Carga Llama 3.2 3B con configuración optimizada"""
    print(f"🤖 Cargando {CONFIG['model_name']}...")
    
    # Cargar tokenizador
    print("📝 Cargando tokenizador...")
    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["model_name"]
    )
    # Configurar pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    print(f"   ✅ Tokenizador cargado. Vocab: {len(tokenizer)}")
    
    # Cargar modelo con configuración optimizada para Kaggle
    print("🧠 Cargando modelo base...")
    
    # Configuración para Kaggle - usar GPU si disponible
    device_map = "auto" if torch.cuda.is_available() else None
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_name"],
        torch_dtype=torch_dtype,
        device_map=device_map,
        trust_remote_code=True,
    )
    
    # Preparar para LoRA si no hay quantización
    if not torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)
    
    print(f"   ✅ Modelo cargado")
    print(f"   💾 Parámetros: {model.num_parameters():,}")
    
    return model, tokenizer

# Cargar modelo
base_model, tokenizer = cargar_modelo_llama32()

🤖 Cargando meta-llama/Llama-3.2-1B-Instruct...
📝 Cargando tokenizador...
   ✅ Tokenizador cargado. Vocab: 128256
🧠 Cargando modelo base...
   ✅ Modelo cargado
   💾 Parámetros: 1,235,814,400


## 8. Aplicación de LoRA

In [22]:
def aplicar_lora(model):
    """Aplica LoRA al modelo base"""
    print("🔧 Aplicando LoRA...")
    
    # Crear configuración LoRA
    lora_config = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"],
        target_modules=LORA_CONFIG["target_modules"],
    )
    
    # Aplicar LoRA
    model_lora = get_peft_model(model, lora_config)
    
    # Estadísticas
    trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model_lora.parameters())
    
    print(f"✅ LoRA aplicado")
    print(f"📊 Parámetros entrenables: {trainable:,} ({trainable/total*100:.2f}%)")
    print(f"📊 Parámetros totales: {total:,}")
    
    # Mostrar módulos LoRA
    print(f"\n🎯 Módulos LoRA activos:")
    lora_modules = [name for name, _ in model_lora.named_modules() if "lora" in name.lower()]
    for module in lora_modules[:5]:  # Mostrar solo los primeros 5
        print(f"   {module}")
    if len(lora_modules) > 5:
        print(f"   ... y {len(lora_modules)-5} más")
    
    return model_lora

# Aplicar LoRA
model = aplicar_lora(base_model)

🔧 Aplicando LoRA...
✅ LoRA aplicado
📊 Parámetros entrenables: 1,179,648 (0.10%)
📊 Parámetros totales: 1,236,994,048

🎯 Módulos LoRA activos:
   base_model.model.model.layers.0.self_attn.q_proj.lora_dropout
   base_model.model.model.layers.0.self_attn.q_proj.lora_dropout.default
   base_model.model.model.layers.0.self_attn.q_proj.lora_A
   base_model.model.model.layers.0.self_attn.q_proj.lora_A.default
   base_model.model.model.layers.0.self_attn.q_proj.lora_B
   ... y 427 más


## 9. Configuración del Entrenamiento

In [ ]:
def crear_training_arguments():
    """Crea argumentos de entrenamiento optimizados"""
    print("⚙️ Configurando entrenamiento...")
    
    # Calcular pasos
    num_samples = len(train_dataset)
    effective_batch_size = CONFIG["batch_size"] * CONFIG["gradient_accumulation"]
    steps_per_epoch = num_samples // effective_batch_size
    max_steps = steps_per_epoch * CONFIG["num_epochs"]
    warmup_steps = int(max_steps * CONFIG["warmup_ratio"])
    
    print(f"📊 Configuración de entrenamiento:")
    print(f"   Muestras: {num_samples}")
    print(f"   Batch efectivo: {effective_batch_size}")
    print(f"   Pasos por época: {steps_per_epoch}")
    print(f"   Pasos totales: {max_steps}")
    print(f"   Warmup steps: {warmup_steps}")
    print(f"   Max sequence length: {CONFIG['max_seq_length']} (controlado por dataset)")
    
    training_args = TrainingArguments(
        # Directorios
        output_dir=CONFIG["output_dir"],
        logging_dir=CONFIG["logs_dir"],
        
        # Entrenamiento
        num_train_epochs=CONFIG["num_epochs"],
        per_device_train_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation"],
        learning_rate=CONFIG["learning_rate"],
        
        # Scheduler
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        
        # Guardado
        save_steps=CONFIG["save_steps"],
        save_strategy=CONFIG["save_strategy"],
        save_total_limit=2,
        logging_steps=CONFIG["logging_steps"],
        eval_steps=CONFIG["eval_steps"],
        eval_strategy=CONFIG["eval_strategy"],
        
        # Optimización para Kaggle
        optim="adamw_8bit",      # Optimizador más eficiente
        weight_decay=0.01,
        max_grad_norm=1.0,
        
        # Precisión - usar fp16 si hay GPU
        fp16=torch.cuda.is_available(),
        bf16=False,
        
        # Otros
        dataloader_drop_last=True,
        remove_unused_columns=False,
        report_to="none",  # Sin logging externo
        seed=42,
    )
    
    return training_args

# Crear argumentos
training_arguments = crear_training_arguments()
print("✅ Argumentos de entrenamiento creados")

⚙️ Configurando entrenamiento...
📊 Configuración de entrenamiento:
   Muestras: 100
   Batch efectivo: 8
   Pasos por época: 12
   Pasos totales: 24
   Warmup steps: 2
✅ Argumentos de entrenamiento creados


## 10. Preparación del Trainer

In [ ]:
def crear_trainer():
    """Crea el SFTTrainer"""
    print("🏃‍♂️ Preparando SFTTrainer...")
    
    # Configuración básica y mínima compatible con trl 0.7.11
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,  # Ahora incluimos eval_dataset
        args=training_arguments,
    )
    
    # Configurar tokenizer manualmente
    trainer.tokenizer = tokenizer
    if trainer.tokenizer.pad_token is None:
        trainer.tokenizer.pad_token = trainer.tokenizer.eos_token
    
    print("✅ SFTTrainer preparado con configuración básica")
    print(f"📦 Train dataset: {len(trainer.train_dataset)} ejemplos")
    print(f"📊 Eval dataset: {len(trainer.eval_dataset)} ejemplos")
    print(f"🔤 Tokenizer configurado manualmente")
    
    return trainer

# Crear trainer
trainer = crear_trainer()

🏃‍♂️ Preparando SFTTrainer...


Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


✅ SFTTrainer preparado
📦 Dataset: 100 ejemplos
🔤 Max length: 256


## 11. ¡ENTRENAMIENTO!

**⚠️ IMPORTANTE:**
- Este proceso puede tomar 1-3 horas
- Monitorea la pérdida (loss) - debe disminuir
- Si hay errores de memoria, reduce `batch_size` o `max_seq_length`

In [25]:
def entrenar():
    """Ejecuta el entrenamiento"""
    print("🚀 INICIANDO ENTRENAMIENTO")
    print("=" * 50)
    
    start_time = datetime.now()
    print(f"⏰ Inicio: {start_time.strftime('%H:%M:%S')}")
    
    try:
        # ¡ENTRENAR!
        result = trainer.train()
        
        end_time = datetime.now()
        duration = end_time - start_time
        
        print("\n🎉 ENTRENAMIENTO COMPLETADO")
        print("=" * 50)
        print(f"⏰ Fin: {end_time.strftime('%H:%M:%S')}")
        print(f"⏱️ Duración: {duration}")
        print(f"📉 Loss final: {result.training_loss:.4f}")
        
        return True, result
        
    except KeyboardInterrupt:
        print("\n⚠️ Entrenamiento interrumpido")
        return False, None
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        return False, None

# ¡EJECUTAR ENTRENAMIENTO!
success, training_result = entrenar()

🚀 INICIANDO ENTRENAMIENTO
⏰ Inicio: 11:03:33


c:\Users\arasa\OneDrive - UTN - Santa Fe\Facultad\5° Año\Proyecto Final\proyecto-final-repo\proyecto_final\.venv\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,2.200700
20,1.949500



🎉 ENTRENAMIENTO COMPLETADO
⏰ Fin: 17:28:12
⏱️ Duración: 6:24:39.461992
📉 Loss final: 2.0280


## 12. Guardar Modelo Entrenado

In [26]:
def guardar_modelo():
    """Guarda el modelo entrenado"""
    if not success:
        print("❌ No se puede guardar - entrenamiento no completado")
        return None
    
    print("💾 Guardando modelo...")
    
    # Directorio final
    final_dir = f"{CONFIG['output_dir']}/final"
    os.makedirs(final_dir, exist_ok=True)
    
    # Guardar modelo LoRA
    model.save_pretrained(final_dir)
    print(f"✅ Modelo LoRA guardado en: {final_dir}")
    
    # Guardar tokenizador
    tokenizer.save_pretrained(final_dir)
    print(f"✅ Tokenizador guardado")
    
    # Guardar configuración
    config_info = {
        "base_model": CONFIG["model_name"],
        "dataset": CONFIG["dataset_name"],
        "num_samples": len(train_dataset),
        "lora_config": LORA_CONFIG,
        "training_loss": training_result.training_loss if training_result else None,
        "date": datetime.now().isoformat(),
    }
    
    with open(f"{final_dir}/training_info.json", "w") as f:
        json.dump(config_info, f, indent=2)
    
    print(f"✅ Información guardada")
    print(f"\n📁 Modelo completo en: {final_dir}")
    
    return final_dir

# Guardar modelo
model_path = guardar_modelo()

💾 Guardando modelo...
✅ Modelo LoRA guardado en: ../models/llama-sql-lora/final
✅ Tokenizador guardado
✅ Información guardada

📁 Modelo completo en: ../models/llama-sql-lora/final


## 13. Prueba del Modelo

In [27]:
def probar_modelo():
    """Prueba el modelo entrenado"""
    if not model_path:
        print("❌ No hay modelo para probar")
        return
    
    print("🧪 PROBANDO MODELO ENTRENADO")
    print("=" * 40)
    
    def generar_sql(schema, question):
        """Genera SQL usando el modelo entrenado"""
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
{schema}

Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
        
        # Tokenizar
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=800)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        # Generar
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>")
            )
        
        # Decodificar
        response = tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        # Extraer SQL generado
        if "<|start_header_id|>assistant<|end_header_id|>" in response:
            sql_part = response.split("<|start_header_id|>assistant<|end_header_id|>")[1]
            sql_part = sql_part.split("<|eot_id|>")[0].strip()
        else:
            sql_part = "Error en generación"
        
        return sql_part
    
    # Ejemplos de prueba
    tests = [
        {
            "schema": "CREATE TABLE users (id INT, name VARCHAR(50), age INT, city VARCHAR(50));",
            "question": "Get all users older than 25 from New York"
        },
        {
            "schema": "CREATE TABLE products (id INT, name VARCHAR(100), price DECIMAL, category VARCHAR(50)); CREATE TABLE orders (id INT, product_id INT, quantity INT, total DECIMAL);",
            "question": "Find total revenue by product category"
        },
        {
            "schema": "CREATE TABLE employees (id INT, name VARCHAR(50), department VARCHAR(50), salary DECIMAL);",
            "question": "What is the average salary per department?"
        }
    ]
    
    for i, test in enumerate(tests, 1):
        print(f"\n🧪 PRUEBA {i}:")
        print(f"Schema: {test['schema'][:60]}...")
        print(f"Pregunta: {test['question']}")
        
        try:
            sql = generar_sql(test["schema"], test["question"])
            print(f"✅ SQL: {sql}")
        except Exception as e:
            print(f"❌ Error: {e}")
        
        print("-" * 40)

# Probar modelo
probar_modelo()

🧪 PROBANDO MODELO ENTRENADO

🧪 PRUEBA 1:
Schema: CREATE TABLE users (id INT, name VARCHAR(50), age INT, city ...
Pregunta: Get all users older than 25 from New York
✅ SQL: SELECT * FROM users WHERE age > 25 AND city = 'New York';
----------------------------------------

🧪 PRUEBA 2:
Schema: CREATE TABLE products (id INT, name VARCHAR(100), price DECI...
Pregunta: Find total revenue by product category
✅ SQL: SELECT SUM(T2.price * T1.quantity) FROM products AS T1 INNER JOIN orders AS T2 ON T1.id = T2.product_id GROUP BY T1.category
----------------------------------------

🧪 PRUEBA 3:
Schema: CREATE TABLE employees (id INT, name VARCHAR(50), department...
Pregunta: What is the average salary per department?
✅ SQL: SELECT AVG(salary) FROM employees GROUP BY department
----------------------------------------


## 14. Script de Integración

In [28]:
def crear_script_uso():
    """Crea script para usar el modelo en tu proyecto"""
    if not model_path:
        print("❌ No hay modelo para crear script")
        return
    
    script = f'''# Script para usar el modelo SQL entrenado
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

class LlamaSQLGenerator:
    def __init__(self, model_path="{model_path}"):
        print("🤖 Cargando modelo SQL Llama 3.2...")
        
        # Cargar modelo base
        self.base_model = AutoModelForCausalLM.from_pretrained(
            "{CONFIG['model_name']}",
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        
        # Cargar adaptadores LoRA
        self.model = PeftModel.from_pretrained(self.base_model, model_path)
        
        # Cargar tokenizador
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        print("✅ Modelo cargado")
    
    def generar_sql(self, schema, question):
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL generator. Convert natural language questions to precise SQL queries based on the provided database schema. Return only the SQL query without explanations.<|eot_id|><|start_header_id|>user<|end_header_id|>

Database Schema:
{{schema}}

Question: {{question}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
        
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=800)
        inputs = {{k: v.to(self.model.device) for k, v in inputs.items()}}
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.1,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        if "<|start_header_id|>assistant<|end_header_id|>" in response:
            sql_part = response.split("<|start_header_id|>assistant<|end_header_id|>")[1]
            sql_part = sql_part.split("<|eot_id|>")[0].strip()
        else:
            sql_part = "Error en generación"
        
        return sql_part

# Función compatible con tu código existente
def generar_sql_con_llama(prompt):
    generator = LlamaSQLGenerator()
    # Parsear prompt simple
    if "Schema:" in prompt and "Question:" in prompt:
        schema = prompt.split("Question:")[0].replace("Schema:", "").strip()
        question = prompt.split("Question:")[1].replace("Return only the SQL query:", "").strip()
    else:
        schema = "Unknown"
        question = prompt
    
    return generator.generar_sql(schema, question)

# Ejemplo de uso
if __name__ == "__main__":
    generator = LlamaSQLGenerator()
    sql = generator.generar_sql(
        "CREATE TABLE users (id INT, name VARCHAR(50), age INT);",
        "Get users older than 25"
    )
    print(f"SQL: {{sql}}")
'''
    
    # Guardar script
    script_path = "../scripts/llama_sql_generator.py"
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(script)
    
    # Instrucciones
    instructions = f'''# CÓMO USAR TU MODELO LLAMA SQL

## En tu run_batch.py:
```python
# Cambiar:
from scripts.generate_sql import generar_sql_con_ollama
# Por:
from scripts.llama_sql_generator import generar_sql_con_llama

# Y usar:
sql_generado = generar_sql_con_llama(prompt)
```

## Uso directo:
```python
from scripts.llama_sql_generator import LlamaSQLGenerator

generator = LlamaSQLGenerator()
sql = generator.generar_sql(schema, question)
```

## Modelo guardado en: {model_path}
'''
    
    with open("../COMO_USAR_LLAMA_SQL.md", "w", encoding="utf-8") as f:
        f.write(instructions)
    
    print(f"✅ Script creado: {script_path}")
    print(f"✅ Instrucciones: ../COMO_USAR_LLAMA_SQL.md")

# Crear scripts
crear_script_uso()

✅ Script creado: ../scripts/llama_sql_generator.py
✅ Instrucciones: ../COMO_USAR_LLAMA_SQL.md


## 🎉 ¡ENTRENAMIENTO COMPLETADO!

### ✅ Lo que has logrado:

1. **Modelo entrenado**: Llama 3.2 3B especializado en SQL
2. **LoRA aplicado**: Entrenamiento eficiente (~1% parámetros)
3. **Datos de calidad**: 1200+ ejemplos filtrados
4. **Modelo guardado**: Listo para usar
5. **Scripts creados**: Integración fácil

### 📁 Archivos generados:

- `../models/llama-sql-lora/final/` - Modelo entrenado
- `../scripts/llama_sql_generator.py` - Script de uso
- `../COMO_USAR_LLAMA_SQL.md` - Instrucciones

### 🚀 Próximos pasos:

1. **Probar más ejemplos** en la celda anterior
2. **Integrar en run_batch.py** usando el script generado
3. **Comparar rendimiento** vs modelo original
4. **Ajustar parámetros** si es necesario

¡Tu modelo Llama 3.2 SQL está listo! 🎯